In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_reviews AS
WITH parsed AS (
  SELECT
    LOWER(TRIM(REGEXP_REPLACE(game_name, '[®™©]', ''))) AS game_key,
    game_name,
    recommendation,
    date,

    -- 4-digit number glued to the front of the review text.
    -- Used ONLY to strip that prefix off review_text -- it is NOT a year.
    CAST(NULLIF(REGEXP_EXTRACT(review, '^(\\d{4})\\s'), '') AS INT) AS text_prefix_num,
    TRIM(REGEXP_REPLACE(review, '^\\d{4}\\s', '')) AS review_text,

    TRY_CAST(REPLACE(hours_played, ',', '') AS DOUBLE) AS hours_played,
    TRY_CAST(REPLACE(helpful, ',', '') AS INT) AS helpful,
    TRY_CAST(REPLACE(funny, ',', '') AS INT) AS funny,

    -- Steam's own date field, two shapes: "November 27, 2019" / "7 December, 2023".
    -- Rows with no year ("March 21") parse to NULL by design.
    COALESCE(
      TRY_TO_DATE(date, 'MMMM d, yyyy'),
      TRY_TO_DATE(date, 'd MMMM, yyyy')
    ) AS review_date,

    NULLIF(TRIM(REGEXP_EXTRACT(username, '^([^\\n]+)')), '') AS username,
    CAST(NULLIF(REGEXP_REPLACE(REGEXP_EXTRACT(username, '([\\d,]+) products'), ',', ''), '') AS INT) AS reviewer_products

  FROM bronze_reviews
  WHERE review IS NOT NULL AND game_name IS NOT NULL
)

SELECT
  game_key,
  game_name,
  recommendation,
  review_text,
  hours_played,
  helpful,
  funny,
  username,
  reviewer_products,
  date                    AS date_raw,
  review_date,
  YEAR(review_date)       AS review_year,
  CASE WHEN review_date IS NOT NULL THEN 1 ELSE 0 END AS has_date
FROM parsed
WHERE TRIM(review_text) <> ''

In [0]:
%sql
SELECT * FROM silver_reviews
WHERE review_date IS NOT NULL
LIMIT 25